In [ ]:
# kaldjian
# 26 may 2021
# splitting DRC DHIS malaria data extracts for ease of use

import os

import pandas as pd

In [ ]:
YEAR = 2026
PROJ_BASE_PATH = "/home/hexa/workspace/pnlp-tdb-pipeline"
OUTPUT_PATH = f"{PROJ_BASE_PATH}" "/data/interim/{}.feather"

In [ ]:
indicator_ids = [
    "CGZbvJchfjk",
    "AxJhIi7tUam",
    "rfeqp2kdOGi",
    "nRm30I4w9En",
    "aZwnLALknnj",
    "CIzQAR8IWH1",
    "SpmQSLRPMl4",
    "kGmdIbd3mbn",
    "pOZcn5GNnRD",
    "fsBcCCKHGUx",
    "uV53nh3MrYl",
    "QafSfDEL1Ku",
    # "cTLKwfG8pSv",  # --> replaced by "pak21wvkWJC" https://bluesquare.atlassian.net/browse/COMACEP-105
    "D3h3Qvl0332",
    "IPnqGkUUN6U",
    "BCcFZzgTlxQ",
    "uIqmxJcTuub",
    "sRbXNrdKvyl",
    "e6qMP9fVtG9",
    "wfmDVt6RVm2",
    "nFjuJLLeejV",
    "aK0QXqm8Zxn",
    "JXm8J8GRJxI",
    "pak21wvkWJC",
]

logistics_ids = [
    "ONRbtRbf64B",
    "pScfYi2Qxop",
    "bwUSX4FMAEM",
    "imqSmhfQTPN",
    "NquDFFwuMRu",
    "hgjnpRWXunB",
    "HMnLiYTxJBf",
    "fHyOLs6mPEr",
    "eounDZJHMgn",
    "FVJ2v5RgBgL",
    "ZpmFCs2dyxD",
    "b0f8cOW1CFV",
    "ZJnypGdwZ0G",
    "Xt48UQAZppa",
    "JD1dcTpKSic",
    "ccpDQ5umc0a",
    "oDOcYJ4wLRP",
    "jlRZE5tTARe",
    "ypN69muDHoO",
    "bdy45DCTJLB",
    "xqw3Jvrkk1M",
    "PLjvTAvttMR",
    "UHAfqWdf7WO",
    "eKHjiGzfBep",
    "YQG37Uw70oU",
    "vtqAD1gqMxx",
]

### ~R~outine

In [ ]:
# if YEAR > 2021:
extract_paths = [
    (f"{PROJ_BASE_PATH}/data/raw/routine/" f"{str(YEAR)}/Q{x}/analytics.csv")
    for x in range(1, 5)
]

extract_paths = [x for x in extract_paths if os.path.exists(x)]

# else:
#     extract_paths = [('/home/jovyan/s3-hexa-cod-msp/data/snis-dhis/'
#                    f'pnlp-indicators/pnlp_indicators_{input_extension}.csv')]

In [ ]:
def read_and_standardize(file_path):
    
    raw = pd.read_csv(file_path)
    
    # standardize column names bewteen extract formats

    routine_df = raw.loc[:, ~raw.columns.str.contains("^Unnamed")].copy()

    if "dx_uid" in routine_df.columns:  # OH classic DHIS extract format
        routine_df.rename(columns={"dx_uid": "de_uid", "dx_name": "de_name"}, inplace=True)
        routine_df.columns = routine_df.columns.str.upper()

        routine_df["OU_NAME"] = routine_df["LEVEL_5_NAME"]
    elif "dx" in routine_df.columns:  # OH workspace format
        routine_df.rename(
            columns={
                "pe": "period",
                "dx": "de_uid",
                "dx_name": "de_name",
                "ou": "ou_uid",
                "co": "coc_uid",
                "co_name": "coc_name",
                "parent_level_1_id": "level_1_uid",
                "parent_level_1_name": "level_1_name",
                "parent_level_2_id": "level_2_uid",
                "parent_level_2_name": "level_2_name",
                "parent_level_3_id": "level_3_uid",
                "parent_level_3_name": "level_3_name",
                "parent_level_4_id": "level_4_uid",
                "parent_level_4_name": "level_4_name",
            },
            inplace=True,
        )
        routine_df.columns = routine_df.columns.str.upper()

        routine_df["LEVEL_5_NAME"] = routine_df["OU_NAME"]
        routine_df["LEVEL_5_UID"] = routine_df["OU_UID"]

    else:
        print("assuming columns are properly formatted")

    # keep data from just that year
    routine_df["YEAR"] = routine_df["PERIOD"].astype(str).str.slice(0, 4)
    routine_df = routine_df.loc[routine_df.YEAR.astype(int) == YEAR]
    
    return(routine_df)

In [ ]:
routine_df = pd.concat([read_and_standardize(p) for p in extract_paths])

In [ ]:
required_cols = [
    "DE_UID",
    "COC_UID",
    "OU_UID",
    "PERIOD",
    "VALUE",
    "DE_NAME",
    "COC_NAME",
    "LEVEL_1_UID",
    "LEVEL_1_NAME",
    "LEVEL_2_UID",
    "LEVEL_2_NAME",
    "LEVEL_3_UID",
    "LEVEL_3_NAME",
    "LEVEL_4_UID",
    "LEVEL_4_NAME",
    "LEVEL_5_UID",
    "LEVEL_5_NAME",
    "YEAR",
    "OU_NAME",
]

assert all(col in routine_df.columns for col in required_cols)

In [ ]:
ind_df = routine_df.loc[routine_df.DE_UID.isin(indicator_ids)].reset_index(drop=True)
ind_df.to_feather(OUTPUT_PATH.format(f"PNLP_indicators_{str(YEAR)}"))
print("indicators written to S3")

log_df = routine_df.loc[routine_df.DE_UID.isin(logistics_ids)].reset_index(drop=True)
log_df.to_feather(OUTPUT_PATH.format(f"PNLP_logistics_{str(YEAR)}"))
print("logistics written to S3")

### All-cause mortality

In [ ]:
extract_path = (
    f"{PROJ_BASE_PATH}/data/raw/" f"all-cause-mortality/{str(YEAR)}/analytics.csv"
)

In [ ]:
raw = pd.read_csv(extract_path)

print("extract in memory, len", len(raw))

In [ ]:
# standardize column names bewteen extract formats

acm = raw.loc[:, ~raw.columns.str.contains("^Unnamed")].copy()

if "dx_uid" in acm.columns:  # OH classic DHIS extract format
    acm.rename(columns={"dx_uid": "ind_uid", "dx_name": "ind_name"}, inplace=True)
    acm.columns = acm.columns.str.upper()

    acm["OU_NAME"] = acm["LEVEL_5_NAME"]
elif "dx" in acm.columns:  # OH workspace format
    acm.rename(
        columns={
            "pe": "period",
            "dx": "ind_uid",
            "dx_name": "ind_name",
            "ou": "ou_uid",
            "co": "coc_uid",
            "co_name": "coc_name",
            "parent_level_1_id": "level_1_uid",
            "parent_level_1_name": "level_1_name",
            "parent_level_2_id": "level_2_uid",
            "parent_level_2_name": "level_2_name",
            "parent_level_3_id": "level_3_uid",
            "parent_level_3_name": "level_3_name",
            "parent_level_4_id": "level_4_uid",
            "parent_level_4_name": "level_4_name",
        },
        inplace=True,
    )
    acm.columns = acm.columns.str.upper()

    acm["LEVEL_5_NAME"] = acm["OU_NAME"]
    acm["LEVEL_5_UID"] = acm["OU_UID"]

else:
    print("assuming columns are properly formatted")

In [ ]:
required_cols = [
    "IND_UID",
    "OU_UID",
    "PERIOD",
    "VALUE",
    "IND_NAME",
    "LEVEL_1_UID",
    "LEVEL_1_NAME",
    "LEVEL_2_UID",
    "LEVEL_2_NAME",
    "LEVEL_3_UID",
    "LEVEL_3_NAME",
    "LEVEL_4_UID",
    "LEVEL_4_NAME",
    "LEVEL_5_UID",
    "LEVEL_5_NAME",
    "OU_NAME",
]

assert all(col in acm.columns for col in required_cols)

In [ ]:
acm.reset_index(drop=True, inplace=True)
acm.to_feather(OUTPUT_PATH.format(f"all_cause_mortality_{str(YEAR)}"))
print("all-cause mortality written to S3")

### Promptitude / complétude

In [ ]:
if YEAR > 2022:
    extract_path = (
        f"{PROJ_BASE_PATH}/data/raw/reporting-rates/"
        f"{YEAR}/analytics.csv"
    )
else:

    if YEAR > 2019:
        year_extension = str(YEAR)
    else:
        year_extension = "2018-2019"

    extract_path = (
        f"{PROJ_BASE_PATH}/data/raw/reporting-rates/"
        f"malaria_data_reporting_rates_{year_extension}.csv"
    )

In [ ]:
raw = pd.read_csv(extract_path)

print("extract in memory, len", len(raw))

In [ ]:
# standardize column names bewteen extract formats
# (may remove this later)

cp = raw.loc[:, ~raw.columns.str.contains("^Unnamed")].copy()

try:
    cp.rename(
        columns={
            "pe": "period",
            "ds": "ds_uid",
            "ou": "ou_uid",
            "parent_level_1_id": "level_1_uid",
            "parent_level_1_name": "level_1_name",
            "parent_level_2_id": "level_2_uid",
            "parent_level_2_name": "level_2_name",
            "parent_level_3_id": "level_3_uid",
            "parent_level_3_name": "level_3_name",
            "parent_level_4_id": "level_4_uid",
            "parent_level_4_name": "level_4_name",
            "metric": "reporting_type"
        },
        inplace=True,
    )
    cp.columns = cp.columns.str.upper()

    cp["LEVEL_5_NAME"] = cp["OU_NAME"]
    cp["LEVEL_5_UID"] = cp["OU_UID"]
except:
    print("Columns properly formatted")

# keep data from just that year
cp["YEAR"] = cp["PERIOD"].astype(str).str.slice(0, 4)
cp = cp.loc[cp.YEAR.astype(int) == YEAR]

In [ ]:
required_cols = [
    "OU_UID",
    "PERIOD",
    "VALUE",
    "DS_UID",
    "REPORTING_TYPE",
    "OU_NAME",
    "LEVEL_1_UID",
    "LEVEL_1_NAME",
    "LEVEL_2_UID",
    "LEVEL_2_NAME",
    "LEVEL_3_UID",
    "LEVEL_3_NAME",
    "LEVEL_4_UID",
    "LEVEL_4_NAME",
    "LEVEL_5_UID",
    "LEVEL_5_NAME",
    "YEAR"
]
assert all(col in cp.columns for col in required_cols)

In [ ]:
cp.reset_index(drop=True, inplace=True)
cp.to_feather(OUTPUT_PATH.format(f"PNLP_DHIS_reporting_rates_{str(YEAR)}"))
print("reporting rates written to S3")